# Piloto de audiolivro adaptado: Reinações de Narizinho

Narra, numa GPU grátis do Google Colab, os capítulos 1 a 3 de *Reinações de Narizinho*, numa adaptação para crianças e adolescentes com deficiência visual, síndrome de Down, autismo e TDAH. O guia de adaptação e de narração está em `notebooks/audiolivro/GUIA.md`.

**Antes de começar:** menu `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4`. Depois rode as células de cima para baixo. Rodar de novo qualquer célula é seguro.

**Saída** (no seu Google Drive, em `MyDrive/Audiolivros/narizinho/capNN/`), para cada capítulo:
- `capNN_A_so_voz.mp3`: só a narração, para quem é sensível a sons;
- `capNN_B_com_ambiente.mp3`: narração com vinheta suave e som de riacho bem baixinho.

Escolha os capítulos na célula 2 (`CAPITULOS_PARA_GERAR`). As vozes ficam em `narizinho/vozes/` e são as mesmas em todos os capítulos.

**Licença:** a voz padrão (OmniVoice) é de uso **não comercial** (CC-BY-NC). Serve para um piloto gratuito. As vozes são sintéticas e originais, sem clonar ninguém.

**Sotaque:** o modelo fala português, mas às vezes puxa para o sotaque de Portugal. Se acontecer, troque a `semente` da voz na célula 5. Outra saída é usar uma gravação de 5 a 10 segundos de alguém que **autorizou por escrito** o uso da própria voz (campo `amostra`).

In [ ]:
# ── 1. Conferir a GPU ───────────────────────────────────────────────────────
import shutil, subprocess
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("ATENÇÃO: sem GPU. Vá em Ambiente de execução -> Alterar o tipo de ambiente "
          "de execução -> GPU T4, e rode o notebook de novo do início.")

In [ ]:
# ── 2. Google Drive e pastas ────────────────────────────────────────────────
import os
from google.colab import drive
drive.mount("/content/drive")

CAPITULOS_PARA_GERAR = [1, 2, 3]   # quais capítulos narrar (ex.: [2, 3])

LIVRO = "/content/drive/MyDrive/Audiolivros/narizinho"
PASTA_VOZES = os.path.join(LIVRO, "vozes")   # uma fala de referência por personagem
os.makedirs(PASTA_VOZES, exist_ok=True)
print("Os arquivos vão para:", LIVRO)

In [ ]:
# ── 3. Instalar o VoiceStudio (primeira vez: ~5-8 min) ──────────────────────
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/sdpaula36-stack/VoiceStudio"
REPO_DIR = "/content/VoiceStudio"
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, *, cwd=None, what=""):
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}")
    if subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode != 0:
        raise SystemExit(f"\nFALHOU: {what or shown}. Veja o erro acima; rodar de novo é seguro.")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    run(["git", "-C", REPO_DIR, "pull", "--ff-only"], what="git pull")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], what="git clone")

run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get install")
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
run(["uv", "pip", "install", "--system", "--no-cache",
     "--constraint", "deploy/torch-constraints.txt",
     "--override", "deploy/uv-overrides.txt", "."],
    cwd=REPO_DIR, what="instalação do backend")

from huggingface_hub import snapshot_download
print("Baixando o modelo de voz (alguns GB, só na primeira vez)...")
snapshot_download("k2-fsa/OmniVoice")
print("Pronto.")

In [ ]:
# ── 4. Ligar o servidor do VoiceStudio ──────────────────────────────────────
import json, os, subprocess, sys, time, urllib.request

PORT = 3900
BASE = f"http://127.0.0.1:{PORT}"
LOG_PATH = "/content/voicestudio_backend.log"

def health():
    try:
        with urllib.request.urlopen(f"{BASE}/health", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = health()
if not info:
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    env = os.environ.copy()
    env.update(OMNIVOICE_SERVER_MODE="1", PYTHONUNBUFFERED="1")
    env.setdefault("OMNIVOICE_DATA_DIR", "/content/omnivoice_data")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app", "--app-dir", "backend",
         "--host", "127.0.0.1", "--port", str(PORT)],
        cwd="/content/VoiceStudio", env=env,
        stdout=open(LOG_PATH, "ab"), stderr=subprocess.STDOUT)
    deadline = time.time() + 300
    while time.time() < deadline and proc.poll() is None and not info:
        print(".", end="", flush=True)
        time.sleep(3)
        info = health()
    print()
    if not info:
        raise SystemExit("O servidor não ligou em 5 min. Final do log:\n"
                         + "".join(open(LOG_PATH, errors="replace").readlines()[-40:]))
print("Servidor ligado:", info)

In [ ]:
# ── 4b. Licenças do modelo de voz ───────────────────────────────────────────
# Leia o que aparece abaixo. Se concordar, troque para True e rode de novo.
ACEITO_AS_LICENCAS = False

import requests
pendentes = []
for repo in ["k2-fsa/OmniVoice", "eustlb/higgs-audio-v2-tokenizer"]:
    st = requests.get(f"{BASE}/models/licenses/acceptance/{repo}", timeout=30).json()
    print(f"{repo}\n  licença: {st.get('license')} ({st.get('category')})\n"
          f"  detalhes: https://huggingface.co/{repo}\n")
    if not st.get("accepted"):
        pendentes.append(st)

if not pendentes:
    print("Licenças já aceitas.")
elif not ACEITO_AS_LICENCAS:
    raise SystemExit("Leia as licenças acima. Se concordar, mude ACEITO_AS_LICENCAS para True "
                     "e rode de novo. 'noncommercial' = o áudio não pode ser vendido.")
else:
    for st in pendentes:
        requests.post(f"{BASE}/models/licenses/accept", timeout=30,
                      json={"repo_id": st["repo_id"], "fingerprint": st["fingerprint"],
                            "accepted": True}).raise_for_status()
        print("Aceita:", st["repo_id"])

## Roteiro

Cada trecho é `[personagem, texto, velocidade, pausa depois em segundos]`.

- **velocidade:** 1.0 é normal; até 1.1 na ação; 0.85 a 0.9 no suspense e no sono.
- **pausa:** o silêncio depois do trecho. Pausas maiores antes das revelações prendem a atenção.
- **voz** aceita só estes termos: `male`/`female`; `child`, `teenager`, `young adult`, `middle-aged`, `elderly`; `very low pitch` … `very high pitch`; `whisper`.
- A voz de cada personagem nasce da fala mais longa dele no **primeiro** capítulo em que aparece, e é a mesma em todos os outros.

Edite à vontade. Os trechos que você mudar são gerados de novo na célula 7.

In [ ]:
# ── 5. Roteiro (EDITE AQUI) ─────────────────────────────────────────────────
# "amostra" (opcional, em cada voz): caminho de um .wav de 5-10 s de alguém que
# AUTORIZOU por escrito o uso da voz. Nunca use a voz de um dublador ou artista.
VOZES = {'narrador': {'voz': 'male, young adult, moderate pitch', 'semente': 201},
 'narizinho': {'voz': 'female, child, high pitch', 'semente': 211},
 'principe': {'voz': 'male, teenager, high pitch', 'semente': 221},
 'major': {'voz': 'male, middle-aged, very low pitch', 'semente': 231},
 'aranha': {'voz': 'female, middle-aged, high pitch', 'semente': 241},
 'caramujo': {'voz': 'male, elderly, low pitch', 'semente': 251},
 'emilia': {'voz': 'female, young adult, very high pitch', 'semente': 261},
 'nastacia': {'voz': 'female, middle-aged, low pitch', 'semente': 271}}

CAPITULOS = {
    1: {"titulo": "Um peixinho no nariz", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo um: Um peixinho no nariz.", 0.95, 1.2],
        ["narrador", "Numa casinha branca, no meio do mato, fica o Sítio do Picapau Amarelo.", 1.0, 0.5],
        ["narrador", "Lá mora uma senhora de óculos, muito boazinha, chamada Dona Benta.", 1.0, 0.5],
        ["narrador", "Dona Benta é avó de uma menina de sete anos. O nome dela é Lúcia.", 1.0, 0.6],
        ["narrador", "Mas ninguém chama a menina de Lúcia. Sabe por quê?", 1.0, 0.8],
        ["narrador", "Porque o nariz dela é pequenininho e virado para cima. Um nariz arrebitado!", 1.05, 0.5],
        ["narrador", "Por isso, todo mundo chama a menina de... Narizinho!", 1.0, 0.9],
        ["narrador", "No sítio também mora Tia Nastácia. Ela faz os melhores bolinhos do mundo e cuida de todos com muito carinho.", 1.0, 0.5],
        ["narrador", "Foi Tia Nastácia quem fez a melhor amiga de Narizinho: uma boneca de pano chamada Emília.", 1.0, 0.5],
        ["narrador", "Emília tem olhos de linha preta e sobrancelhas tão lá no alto que parece estar sempre espantada.", 1.0, 0.6],
        ["narrador", "Só tem um problema. Emília não fala. Nem uma palavrinha.", 0.95, 0.8],
        ["narrador", "Pelo menos... por enquanto.", 0.9, 1.4],
        ["narrador", "Atrás da casa passa um riacho de água bem clarinha. Narizinho adora ir lá.", 1.0, 0.5],
        ["narrador", "Toda tarde ela pega Emília no colo, senta na beira da água e joga migalhas de pão para os peixinhos.", 1.0, 0.5],
        ["narrador", "Os peixinhos já conhecem a menina. Quando ela chega, eles vêm nadando, todos juntos: plic, plic, plic!", 1.05, 0.8],
        ["narrador", "Naquela tarde, o sol estava quentinho. O vento balançava as folhas, bem devagar...", 0.9, 0.6],
        ["narrador", "E Narizinho foi ficando com sono... com sono...", 0.85, 0.8],
        ["narrador", "Até que fechou os olhos e dormiu.", 0.85, 1.8],
        ["narrador", "De repente, alguma coisa fez cócegas no nariz dela!", 1.1, 0.5],
        ["narrador", "Era um peixinho! Um peixinho pequeno, com roupa de escamas brilhantes e uma coroa dourada na cabeça.", 1.05, 0.5],
        ["narrador", "E sabe o que ele estava fazendo? Estava em pé, bem em cima do nariz de Narizinho!", 1.05, 0.7],
        ["principe", "Que montanha mais esquisita! É macia, é quentinha... e tem dois buracos que sopram vento!", 1.0, 0.7],
        ["narrador", "Narizinho ficou quietinha, quietinha, segurando a risada.", 0.95, 0.6],
        ["narrador", "Então, do meio da água, saiu um caranguejo enorme, com as garras abertas. Era o Major Agarra-e-Não-Larga-Mais, o guarda do príncipe.", 1.0, 0.5],
        ["major", "Alteza! Cuidado! Essa montanha pode ser perigosa!", 1.0, 0.5],
        ["principe", "Bobagem, Major. Montanha não faz mal a ninguém.", 1.0, 0.7],
        ["narrador", "Nessa hora, Narizinho não aguentou mais... e espirrou! Atchim!", 1.1, 0.4],
        ["narrador", "O peixinho saiu voando e caiu de costas na grama!", 1.1, 0.4],
        ["principe", "Socorro! A montanha está viva!", 1.1, 0.6],
        ["narizinho", "Calma! Eu não sou montanha. Eu sou uma menina! Meu nome é Narizinho.", 1.0, 0.7],
        ["narrador", "O peixinho se levantou, arrumou a coroa e fez uma reverência bem educada.", 1.0, 0.5],
        ["principe", "Muito prazer, menina. Eu sou o Príncipe Escamado, rei do Reino das Águas Claras.", 0.95, 0.4],
        ["principe", "Desculpe ter subido no seu nariz. Ele é tão bonitinho que eu pensei que fosse uma ilha!", 1.0, 0.7],
        ["narrador", "Narizinho riu. Ninguém nunca tinha elogiado o nariz dela daquele jeito.", 1.0, 0.6],
        ["narizinho", "E onde fica esse tal Reino das Águas Claras?", 1.0, 0.5],
        ["principe", "Lá no fundo do riacho. Tem palácio de conchas, peixes de todas as cores e muitas coisas que você nunca viu.", 1.0, 0.4],
        ["principe", "Quer conhecer?", 0.95, 1.0],
        ["narrador", "Narizinho olhou para a água. Depois olhou para Emília. Emília, como sempre, não disse nada.", 0.95, 0.6],
        ["narizinho", "Eu quero! Mas a Emília vai junto.", 1.05, 0.5],
        ["principe", "Então segure a minha nadadeira. E não solte por nada!", 1.05, 0.6],
        ["narrador", "Narizinho segurou a nadadeira do príncipe, apertou Emília contra o peito...", 1.0, 0.4],
        ["narrador", "E, tchibum! Mergulhou no riacho!", 1.1, 1.6],
        ["narrador", "Lá embaixo, tudo era azul e brilhante. E no palácio de conchas, Narizinho ia conhecer um médico muito esquisito: o Doutor Caramujo.", 0.9, 0.6],
        ["narrador", "O Doutor Caramujo guardava umas pílulas mágicas. E uma dessas pílulas ia mudar a vida da Emília para sempre...", 0.85, 1.0],
        ["narrador", "Mas isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    2: {"titulo": "O palácio de conchas", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo dois: O palácio de conchas.", 0.95, 1.2],
        ["narrador", "No capítulo passado, Narizinho dormiu na beira do riacho e acordou com um peixinho de coroa em cima do nariz.", 1.0, 0.4],
        ["narrador", "Era o Príncipe Escamado. Ele convidou Narizinho para conhecer o Reino das Águas Claras. E ela mergulhou, com Emília no colo!", 1.0, 1.2],
        ["narrador", "Debaixo da água, Narizinho percebeu uma coisa incrível. Ela conseguia respirar! E nem estava molhada!", 1.05, 0.6],
        ["narrador", "Em volta dela passavam peixes amarelos, peixes azuis, peixes listrados. Uns grandes, outros do tamanho de um grão de feijão.", 1.0, 0.5],
        ["narrador", "E lá na frente, brilhando como um castelo de festa, estava o palácio de conchas.", 0.95, 0.8],
        ["narizinho", "Que lindo! Parece feito de pérolas!", 1.05, 0.6],
        ["principe", "Seja bem-vinda ao meu palácio, Narizinho. Aqui você é a minha convidada de honra.", 1.0, 0.7],
        ["narrador", "Na porta do palácio, quem estava de guarda? Ele mesmo: o Major Agarra-e-Não-Larga-Mais, com as garras para cima.", 1.0, 0.4],
        ["major", "Alto lá! Quem vem com o príncipe pode passar. Mas a boneca de pano fica de olho fechado!", 1.0, 0.6],
        ["narrador", "Narizinho riu. Emília não podia fechar os olhos, porque eles eram costurados com linha preta.", 1.0, 0.9],
        ["narrador", "Lá dentro, o príncipe levou Narizinho até uma salinha cheia de fios e agulhas.", 1.0, 0.4],
        ["narrador", "Era o ateliê da Dona Aranha, a costureira mais famosa do reino. Ela tinha oito patas, e cada pata segurava uma agulha diferente.", 1.0, 0.5],
        ["aranha", "Ora, ora, uma menina de verdade! Fique paradinha, que eu vou fazer um vestido de festa para você.", 1.0, 0.6],
        ["narrador", "E as oito patas começaram a costurar, depressa, depressa, depressa! Zip, zip, zip!", 1.1, 0.5],
        ["narrador", "Em um minuto, o vestido estava pronto. Era azul como o céu, com bolinhas de espuma do mar.", 1.0, 0.5],
        ["narizinho", "Dona Aranha, é o vestido mais bonito que eu já vi!", 1.05, 0.5],
        ["aranha", "Fiz com fio de teia e luz da lua. Cuidado para não sujar de lama!", 1.0, 1.0],
        ["narrador", "Depois, o príncipe disse que queria apresentar um amigo muito importante.", 1.0, 0.4],
        ["narrador", "Bateram numa porta pequenininha. Toc, toc, toc.", 0.95, 0.8],
        ["narrador", "A porta abriu bem devagar... e apareceu um caramujo de óculos, com uma maleta preta e cara de sabido.", 0.9, 0.5],
        ["caramujo", "Boa tarde, boa tarde. Eu sou o Doutor Caramujo, o médico do reino. Quem está doente aqui?", 0.95, 0.6],
        ["principe", "Ninguém está doente, doutor. Mas a boneca da Narizinho não fala.", 1.0, 0.5],
        ["caramujo", "Não fala? Hum... Deixe-me ver.", 0.95, 0.6],
        ["narrador", "O doutor olhou a boca de Emília com uma lupa. Olhou de um lado. Olhou do outro.", 0.95, 0.5],
        ["caramujo", "Já sei o que é. Esta boneca tem falta de fala. Mas eu tenho o remédio certo!", 1.0, 0.6],
        ["narrador", "Ele abriu a maleta preta e tirou um vidrinho cheio de bolinhas coloridas.", 1.0, 0.4],
        ["caramujo", "São as minhas pílulas falantes. Uma só, e qualquer um fala pelos cotovelos!", 1.0, 0.8],
        ["narrador", "Narizinho segurou Emília bem firme. O doutor pegou uma pílula e colocou na boca da boneca.", 0.95, 0.5],
        ["narrador", "Emília engoliu. Gulp!", 0.95, 1.2],
        ["narrador", "Todo mundo ficou olhando. O príncipe, a Dona Aranha, o Major, o doutor... e Narizinho, de olhos arregalados.", 0.9, 0.8],
        ["narrador", "Um segundo. Dois segundos. Três segundos...", 0.85, 1.0],
        ["narrador", "E então a boca de pano da Emília começou a mexer.", 0.85, 1.2],
        ["narrador", "Qual será a primeira palavra da Emília? Você vai descobrir no próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    3: {"titulo": "Emília fala!", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo três: Emília fala!", 0.95, 1.2],
        ["narrador", "No capítulo passado, Narizinho ganhou um vestido da Dona Aranha e conheceu o Doutor Caramujo.", 1.0, 0.4],
        ["narrador", "O doutor deu para a Emília uma pílula falante. E a boca de pano da boneca começou a mexer...", 0.95, 1.2],
        ["emilia", "Estou com um gosto horrível de sapo na boca!", 1.05, 0.9],
        ["narrador", "Todo mundo levou um susto. A Emília estava falando!", 1.05, 0.5],
        ["narizinho", "Emília! Você fala! Você fala de verdade!", 1.1, 0.5],
        ["emilia", "Claro que falo. Eu sempre pensei um monte de coisas. Só faltava a boca funcionar!", 1.05, 0.6],
        ["narrador", "E a Emília não parou mais. Falava, falava, falava, igual a uma torneira aberta.", 1.05, 0.5],
        ["emilia", "Esse príncipe é muito molhado. Esse caranguejo é muito bravo. E esse doutor tem cara de quem come alface no café da manhã!", 1.1, 0.6],
        ["caramujo", "Hum. Acho que a pílula era forte demais.", 0.95, 0.6],
        ["narrador", "Narizinho ficou um pouco envergonhada. Mas também não conseguia parar de rir.", 1.0, 0.5],
        ["narizinho", "Emília, fale com educação! Eles são nossos amigos.", 1.0, 0.5],
        ["emilia", "Está bem, está bem. Muito obrigada pela pílula, senhor doutor. Mas da próxima vez, faça com gosto de goiabada!", 1.05, 0.9],
        ["narrador", "O príncipe achou tudo muito engraçado e resolveu fazer uma festa.", 1.0, 0.4],
        ["principe", "Hoje tem baile no palácio! Para comemorar a boneca que aprendeu a falar!", 1.05, 0.6],
        ["narrador", "Os peixes tocaram música com as bolhas: blub, blub, blub. Os camarões dançaram em fila. E até o Major mexeu as garras no ritmo.", 1.05, 0.5],
        ["narrador", "Narizinho dançou com o príncipe, de vestido azul. E Emília dançou sozinha, girando, girando, e falando sem parar.", 1.0, 0.9],
        ["narrador", "No meio da festa, de repente, Narizinho ouviu uma voz que vinha lá de cima. De muito longe.", 0.9, 0.8],
        ["nastacia", "Narizinho! Vem lanchar, menina! Tem bolinho quentinho!", 0.95, 0.8],
        ["narizinho", "É a Tia Nastácia! Preciso voltar para casa.", 1.0, 0.5],
        ["principe", "Já vai? Mas a festa mal começou!", 1.0, 0.5],
        ["narizinho", "Eu volto, prometo. Obrigada por tudo, príncipe!", 1.0, 0.8],
        ["narrador", "Narizinho fechou os olhos. Quando abriu de novo... estava deitada na grama, na beira do riacho.", 0.9, 0.8],
        ["narrador", "Será que tinha sido tudo um sonho?", 0.85, 1.0],
        ["narrador", "Ela olhou para a boneca no colo. E a boneca olhou de volta para ela.", 0.9, 0.8],
        ["emilia", "Que foi? Nunca viu boneca falar, não?", 1.05, 1.0],
        ["narrador", "Não tinha sido sonho! Narizinho saiu correndo para casa, com a Emília falando sem parar.", 1.1, 0.5],
        ["narrador", "E o que será que a Dona Benta e a Tia Nastácia vão dizer quando ouvirem a Emília falar?", 0.95, 0.6],
        ["narrador", "Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
}

for n, c in CAPITULOS.items():
    palavras = sum(len(t[1].split()) for t in c["trechos"])
    print(f"Capítulo {n}, {c['titulo']}: {len(c['trechos'])} trechos, cerca de "
          f"{palavras / 130 + sum(t[3] for t in c['trechos']) / 60:.1f} min.")

In [ ]:
# ── 6. Criar as vozes e ouvir (troque a semente de quem não ficou bom) ──────
import hashlib, os, shutil, requests
from IPython.display import Audio, display

def gerar(texto, destino, **campos):
    arquivos = campos.pop("files", None)
    dados = {"text": texto, "language": "pt", "effect_preset": "raw", **campos}
    r = requests.post(f"{BASE}/generate", data=dados, files=arquivos, timeout=1800)
    if r.status_code != 200:
        raise SystemExit(f"Erro {r.status_code} em /generate: {r.text[:500]}\nLog: {LOG_PATH}")
    with open(destino, "wb") as f:
        f.write(r.content)

def chave(*partes):
    return hashlib.sha1(repr(partes).encode()).hexdigest()[:8]

REF = {}   # personagem -> (wav de referência, texto)
for nome, v in VOZES.items():
    # A fala mais longa do personagem no primeiro capítulo em que ele aparece.
    texto = next(max(falas, key=len) for _, c in sorted(CAPITULOS.items())
                 if (falas := [t[1] for t in c["trechos"] if t[0] == nome]))
    arq = f"{nome}_{chave(v, texto)}.wav"
    wav = os.path.join(PASTA_VOZES, arq)
    antigo = os.path.join(LIVRO, "cap01", "vozes", arq)   # versão do primeiro teste
    if not os.path.exists(wav) and os.path.exists(antigo):
        shutil.copy(antigo, wav)
    if not os.path.exists(wav):
        if v.get("amostra"):
            with open(v["amostra"], "rb") as a:
                gerar(texto, wav, seed=v["semente"], files={"ref_audio": ("amostra.wav", a)})
        else:
            gerar(texto, wav, instruct=v["voz"], seed=v["semente"])
    REF[nome] = (wav, texto)
    print(f"{nome} ({v['voz']}, semente {v['semente']}): {texto}")
    display(Audio(wav))

In [ ]:
# ── 7. Narrar e montar cada capítulo (A: só voz; B: com vinheta e riacho) ───
import os, subprocess
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

def narrar(n):
    pasta = os.path.join(LIVRO, f"cap{n:02d}", "trechos")
    os.makedirs(pasta, exist_ok=True)
    arquivos = []
    for i, (quem, texto, vel, pausa) in enumerate(CAPITULOS[n]["trechos"], 1):
        ref_wav, ref_texto = REF[quem]
        wav = os.path.join(pasta, f"{i:03d}_{quem}_{chave(os.path.basename(ref_wav), texto, vel)}.wav")
        if not os.path.exists(wav):
            with open(ref_wav, "rb") as ref:
                gerar(texto, wav, seed=VOZES[quem]["semente"], ref_text=ref_texto, speed=vel,
                      files={"ref_audio": ("ref.wav", ref, "audio/wav")})
            print(f"{n}.{i:03d} {quem}: {texto}")
        arquivos.append((wav, pausa))
    return arquivos

def tom(freq, dur, vol, sr):
    t = np.arange(int(dur * sr)) / sr
    return (vol * np.sin(2 * np.pi * freq * t) * np.exp(-3 * t)).astype("float32")

def vinheta(sr):
    # Três notas suaves (dó, mi, sol), sem ataque forte.
    v = np.zeros(int(2.2 * sr), dtype="float32")
    for k, f in enumerate([523.25, 659.25, 783.99]):
        nota = tom(f, 1.4, 0.12, sr)
        ini = int(k * 0.35 * sr)
        v[ini:ini + len(nota)] += nota
    return v

def riacho(n, sr):
    # Ruído suave filtrado, que lembra água correndo, bem baixo e com entrada lenta.
    rng = np.random.default_rng(7)
    r = np.convolve(rng.standard_normal(n + 400).astype("float32"),
                    np.ones(40, dtype="float32") / 40, mode="valid")[:n]
    r *= 0.01 * (1 + 0.3 * np.sin(2 * np.pi * 0.15 * np.arange(n) / sr))
    fade = min(n // 2, 4 * sr)
    r[:fade] *= np.linspace(0, 1, fade)
    r[-fade:] *= np.linspace(1, 0, fade)
    return r

def exportar(audio, sr, pasta, nome):
    tmp = os.path.join(pasta, nome + ".wav")
    sf.write(tmp, audio, sr)
    mp3 = os.path.join(pasta, nome + ".mp3")
    # Volume normalizado para fone e caixinha: -16 LUFS, pico máximo -1.5 dBTP.
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp,
                    "-af", "loudnorm=I=-16:TP=-1.5:LRA=11", "-ar", "44100",
                    "-codec:a", "libmp3lame", "-q:a", "2", mp3], check=True)
    print(f"{nome}.mp3: {len(audio) / sr / 60:.1f} min")
    display(Audio(mp3))

def montar(n, arquivos):
    partes, sr = [], None
    for wav, pausa in arquivos:
        a, s = sf.read(wav, dtype="float32", always_2d=True)
        sr = sr or s
        assert s == sr, f"taxa de amostragem diferente em {wav}"
        partes += [a.mean(axis=1), np.zeros(int(pausa * sr), dtype="float32")]
    voz = np.concatenate(partes)
    um_seg = np.zeros(sr, dtype="float32")
    pasta = os.path.join(LIVRO, f"cap{n:02d}")
    exportar(np.concatenate([um_seg, voz, um_seg]), sr, pasta, f"cap{n:02d}_A_so_voz")
    exportar(np.concatenate([vinheta(sr), um_seg, voz + riacho(len(voz), sr), um_seg, vinheta(sr)]),
             sr, pasta, f"cap{n:02d}_B_com_ambiente")

REFAZER = False   # True: narra de novo capítulos que já têm os dois MP3 prontos

for n in CAPITULOS_PARA_GERAR:
    print(f"\n=== Capítulo {n}: {CAPITULOS[n]['titulo']} ===")
    prontos = [os.path.join(LIVRO, f"cap{n:02d}", f"cap{n:02d}_{v}.mp3")
               for v in ("A_so_voz", "B_com_ambiente")]
    if not REFAZER and all(map(os.path.exists, prontos)):
        print("Já estava pronto (para refazer, use REFAZER = True).")
        continue
    montar(n, narrar(n))
print("\nPronto! Os arquivos estão no Drive, em", LIVRO)

# Se um trecho ficou ruim, apague o .wav dele na pasta capNN/trechos/ e rode esta célula de novo.